# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [3]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    'p': ['p0', 'p1'],
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,s0,s1,s2,s3,s_0,s_1,s_2,s_3,p0,p1
878,2,43,"(0.0242491433958297, 0.9266927708557562)","(0.01, -0.045, -0.044, 0.019)",0,1.0,"(0.009, -0.233, -0.043, 0.161)",1.0,1.0,"(0.004, -0.044, -0.04, 0.004)",0.010,-0.045,-0.044,0.019,0.009,-0.233,-0.043,0.161,0.024249,0.926693
709,15,35,"(0.4791648472834677, 0.9955573111901916)","(-0.069, -0.589, 0.142, 0.923)",0,1.0,"(-0.08, -0.781, 0.16, 1.153)",0.0,1.0,"(-0.096, -0.973, 0.183, 1.385)",-0.069,-0.589,0.142,0.923,-0.080,-0.781,0.160,1.153,0.479165,0.995557
994,22,48,"(0.0516687482818331, 0.4972984817210787)","(0.089, 1.174, -0.105, -1.61)",0,1.0,"(0.113, 0.981, -0.137, -1.36)",0.0,1.0,"(0.132, 0.788, -0.164, -1.12)",0.089,1.174,-0.105,-1.610,0.113,0.981,-0.137,-1.360,0.051669,0.497298
687,8,34,"(0.1912497154781393, 0.2075668863455732)","(-0.043, 0.473, -0.03, -1.572)",0,1.0,"(-0.034, 0.258, -0.062, -0.841)",1.0,1.0,"(-0.029, 0.476, -0.079, -1.645)",-0.043,0.473,-0.030,-1.572,-0.034,0.258,-0.062,-0.841,0.191250,0.207567
1777,11,89,"(0.1964578206703505, 1.7230322017264583)","(-0.005, -0.16, 0.024, 0.066)",1,1.0,"(-0.008, 0.026, 0.025, -0.023)",1.0,1.0,"(-0.008, 0.212, 0.025, -0.112)",-0.005,-0.160,0.024,0.066,-0.008,0.026,0.025,-0.023,0.196458,1.723032


# Predict 

In [4]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [5]:
def reagroup(prediction_dataset):
    # agg_results = prediction_dataset[['episode'] + [
    #     f'rse_model_{i}' for i,_ in enumerate(models)
    # ]].groupby('episode').mean().reset_index()
    agg_results = prediction_dataset[['episode'] + [
        f'rse_model_{i}' for i,_ in enumerate(models)
    ]].copy()
    
    agg_results['best_model'] = agg_results.apply(lambda row: np.argmin(row[1:].values), axis=1)
    print(agg_results['best_model'].value_counts())

    prediction_dataset['best_model'] = prediction_dataset.apply(lambda row: agg_results[agg_results['episode']==row['episode']].best_model.values[0],axis=1)
    prediction_dataset['best_rse'] = prediction_dataset.apply(lambda row: row[f'rse_model_{row.best_model}'],axis=1)

    return prediction_dataset

In [6]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3']] = pre_df[['s0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3']]

    prediction_dataset = evaluate(models, pre_df)
    prediction_dataset = reagroup(prediction_dataset)
    final_predictions = evaluate(models, df)
    final_predictions['group'] = prediction_dataset['best_model']

    cols = [
        'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ]

    for c in cols:
        final_predictions[c] = final_predictions.apply(lambda x: x[f'{c}_model_{x.group}'], axis=1)

    return final_predictions[cols]

In [7]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

best_model
3    833
1    522
2    315
0    309
4    148
Name: count, dtype: int64


,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
878,"(0.001, -0.049, -0.039, 0.043)",0.048,0.504498,0.003,0.005,0.001,0.039,0.549102,0.527286,0.503597,0.438008
709,"(-0.1, -0.968, 0.183, 1.578)",0.202,0.507457,0.004,0.005,0.000,0.193,0.550158,0.527286,0.501199,0.451185
994,"(0.132, 0.694, -0.142, -1.18)",0.176,0.522215,0.000,0.094,0.022,0.060,0.545935,0.549164,0.553957,0.439805
687,"(-0.032, 0.447, -0.084, -1.02)",0.662,0.520907,0.003,0.029,0.005,0.625,0.549102,0.533186,0.513189,0.488149
1777,"(-0.007, 0.219, 0.025, -0.246)",0.142,0.505526,0.001,0.007,0.000,0.134,0.546990,0.527778,0.501199,0.446137
